[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asthakalita22/deepfake-detection/blob/main/deepfake_detection.ipynb)

# Deepfake Detection (FaceForensics++ C23)

Pipeline: face-cropped individual frames -> leakage-free split by identity -> pretrained EfficientNetB0 (2-phase fine-tuning) -> threshold chosen on validation -> frame-level and video-level evaluation on test.

**Setup:** add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab (left sidebar, key icon -> Secrets) and enable notebook access. Never paste `kaggle.json` into a cell.

In [ ]:
!pip install -q kagglehub opencv-python scikit-learn tqdm

In [ ]:
import os, glob, re, json, random
import cv2, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = 224
QUICK = True   # True: ~25-30 min total on a Colab GPU. False: bigger dataset, better accuracy, slower.
FRAMES_PER_VIDEO = 6 if QUICK else 10
BATCH = 32
FRAMES_DIR = "/content/frames"
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Download dataset (credentials from Colab secrets)

In [ ]:
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
path = kagglehub.dataset_download("xdxd003/ff-c23")
data_dir = os.path.join(path, "FaceForensics++_C23")
print(data_dir, os.listdir(data_dir))

## 2. Build video list and a leakage-free split

FF++ fakes are named `<a>_<b>.mp4`, built from a pair of original videos. Linking ids through **all** fake filenames (union-find) recovers those pairs, so each pair (its 2 originals and every fake made from them) is one group and goes entirely to one split. 
DeepFakeDetection clips are grouped by their first actor id. Groups are assigned 70/15/15 separately for each family, then videos are sampled per category within each split, so val and test always contain both real and fake.

In [ ]:
FAKE_CATS = ["DeepFakeDetection", "Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]
MAX_PER_FAKE_CAT = 80 if QUICK else 250   # FF++ has ~1000 per category
MAX_REAL = 400 if QUICK else 1000
FRAC = {"train": 0.70, "val": 0.15, "test": 0.15}

def list_videos(cat):
    return sorted(glob.glob(os.path.join(data_dir, cat, "*.mp4")))

# union-find over every FF++ fake filename -> source pairs
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for cat in FAKE_CATS:
    if cat == "DeepFakeDetection":
        continue
    for v in list_videos(cat):
        ids = re.findall(r"\d+", os.path.splitext(os.path.basename(v))[0])
        for i in ids[1:]:
            parent[find(ids[0])] = find(i)

def group_of(cat, stem):
    if cat == "DeepFakeDetection":
        return "dfd_" + stem.split("__")[0].split("_")[0]
    return "ff_" + find(re.findall(r"\d+", stem)[0])

rows = []
for cat in ["original"] + FAKE_CATS:
    for v in list_videos(cat):
        stem = os.path.splitext(os.path.basename(v))[0]
        rows.append(dict(video=v, category=cat, label=0 if cat == "original" else 1,
                         group=group_of(cat, stem)))
all_vids = pd.DataFrame(rows)

# assign groups to splits, separately for each family
group_split = {}
for fam in ["ff_", "dfd_"]:
    gs = sorted(g for g in all_vids["group"].unique() if g.startswith(fam))
    random.Random(SEED).shuffle(gs)
    for k, g in enumerate(gs):
        group_split[g] = "train" if k < 0.70 * len(gs) else "val" if k < 0.85 * len(gs) else "test"
all_vids["split"] = all_vids["group"].map(group_split)

# sample per category within each split
parts = []
for cat in ["original"] + FAKE_CATS:
    cap = MAX_REAL if cat == "original" else MAX_PER_FAKE_CAT
    d = all_vids[all_vids.category == cat]
    for sp, f in FRAC.items():
        ds_ = d[d.split == sp]
        parts.append(ds_.sample(min(len(ds_), int(cap * f)), random_state=SEED))
videos = pd.concat(parts).drop(columns="group").reset_index(drop=True)
print(videos.groupby(["split", "label"]).size().unstack())

## 3. Extract face-cropped frames

Individual frames (not averaged), face detected with OpenCV's Haar cascade, box expanded by 30%, falls back to a centre crop if no face is found. Filenames include the category so nothing gets overwritten.

In [ ]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def crop_face(frame, margin=0.3):
    h, w = frame.shape[:2]
    scale = 480 / max(h, w)
    small = cv2.resize(frame, None, fx=scale, fy=scale) if scale < 1 else frame
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
    if len(faces) == 0:
        s = min(h, w); y0, x0 = (h - s) // 2, (w - s) // 2
        return frame[y0:y0 + s, x0:x0 + s], False
    x, y, fw, fh = max(faces, key=lambda f: f[2] * f[3])
    x, y, fw, fh = [int(v / min(scale, 1)) for v in (x, y, fw, fh)]
    cx, cy, side = x + fw // 2, y + fh // 2, int(max(fw, fh) * (1 + margin))
    x0, y0 = max(cx - side // 2, 0), max(cy - side // 2, 0)
    return frame[y0:y0 + side, x0:x0 + side], True

def extract(row):
    out = []
    stem = os.path.splitext(os.path.basename(row.video))[0]
    cap = cv2.VideoCapture(row.video)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < FRAMES_PER_VIDEO:
        cap.release(); return out
    targets = set(np.linspace(0, total - 1, FRAMES_PER_VIDEO).astype(int).tolist())
    last = max(targets)
    for i in range(last + 1):
        if not cap.grab():
            break
        if i in targets:
            ok, frame = cap.retrieve()
            if not ok:
                continue
            face, found = crop_face(frame)
            if face.size == 0:
                continue
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
            d = os.path.join(FRAMES_DIR, row.split, str(row.label))
            os.makedirs(d, exist_ok=True)
            p = os.path.join(d, f"{row.category}_{stem}_{i:05d}.jpg")
            cv2.imwrite(p, face, [cv2.IMWRITE_JPEG_QUALITY, 95])
            out.append(dict(path=p, label=row.label, split=row.split, video=row.video,
                            category=row.category, face_found=found))
    cap.release()
    return out

meta_csv = os.path.join(FRAMES_DIR, "frames.csv")
def cache_ok(df):
    m = df[["video", "split"]].drop_duplicates().merge(videos[["video", "split"]], on="video",
                                                       how="left", suffixes=("", "_new"))
    return bool((m["split"] == m["split_new"]).all())

if os.path.exists(meta_csv) and cache_ok(pd.read_csv(meta_csv)):
    frames = pd.read_csv(meta_csv)
else:
    import shutil; shutil.rmtree(FRAMES_DIR, ignore_errors=True)
    os.makedirs(FRAMES_DIR, exist_ok=True)
    with ThreadPoolExecutor(max_workers=8) as ex:
        results = list(tqdm(ex.map(extract, videos.itertuples()), total=len(videos)))
    frames = pd.DataFrame([r for rs in results for r in rs])
    frames.to_csv(meta_csv, index=False)

print(frames.groupby(["split", "label"]).size().unstack())
print("face detected in %.1f%% of frames" % (100 * frames["face_found"].mean()))

In [ ]:
# sanity check: a few training crops
import matplotlib.pyplot as plt
sample = frames[frames.split == "train"].groupby("label").sample(4, random_state=SEED)
fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, r) in zip(axs.ravel(), sample.iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{'fake' if r.label else 'real'} / {r.category}", fontsize=9); ax.axis("off")
plt.show()

## 4. Input pipeline

In [ ]:
def make_ds(df, train):
    paths = df["path"].values
    labels = df["label"].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if train:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    def load(p, y):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        return img, y            # EfficientNet expects raw 0-255 pixels
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH)
    return ds.prefetch(tf.data.AUTOTUNE)

train_df = frames[frames.split == "train"].reset_index(drop=True)
val_df   = frames[frames.split == "val"].reset_index(drop=True)
test_df  = frames[frames.split == "test"].reset_index(drop=True)
train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)

counts = np.bincount(train_df["label"])
class_weight = {0: len(train_df) / (2 * counts[0]), 1: len(train_df) / (2 * counts[1])}
print("class_weight:", class_weight)

## 5. Model: pretrained EfficientNetB0

Phase 1 trains only the new head with the backbone frozen. Phase 2 unfreezes the top of the backbone (BatchNorm layers stay frozen) and fine-tunes at a low learning rate.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="augment")

base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

METRICS = [keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.AUC(name="auc")]
CKPT = "/content/best.weights.h5"

def callbacks():
    return [
        keras.callbacks.ModelCheckpoint(CKPT, monitor="val_auc", mode="max",
                                        save_best_only=True, save_weights_only=True),
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
    ]

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05), metrics=METRICS)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=4 if QUICK else 8,
               class_weight=class_weight, callbacks=callbacks())

In [ ]:
# Phase 2: fine-tune the top of the backbone
base.trainable = True
N_UNFREEZE = 40 if QUICK else 100
for layer in base.layers[:-N_UNFREEZE]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05), metrics=METRICS)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=8 if QUICK else 20,
               class_weight=class_weight, callbacks=callbacks())

In [ ]:
# training curves
hist = {k: h1.history[k] + h2.history[k] for k in h1.history if k in h2.history}
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axs, ["loss", "acc", "auc"]):
    ax.plot(hist[k], label="train"); ax.plot(hist["val_" + k], label="val")
    ax.axvline(len(h1.history["loss"]) - 0.5, ls="--", c="gray"); ax.set_title(k); ax.legend()
plt.show()

## 6. Evaluation

The decision threshold is chosen on the **validation** set (max F1) and then applied unchanged to the test set. Video-level scores average the frame probabilities of each video.

In [ ]:
def predict_tta(ds):
    # average of the image and its horizontal mirror
    flipped = ds.map(lambda x, y: (tf.image.flip_left_right(x), y))
    return (model.predict(ds).ravel() + model.predict(flipped).ravel()) / 2

val_prob  = predict_tta(val_ds)
test_prob = predict_tta(test_ds)

ts = np.linspace(0.05, 0.95, 91)
best_t = ts[np.argmax([f1_score(val_df["label"], val_prob > t) for t in ts])]
print(f"threshold chosen on val: {best_t:.2f}")

def report(y, p, t, title):
    pred = (p > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    print(f"--- {title} ---")
    print(f"Acc {accuracy_score(y, pred):.4f} | Prec {precision_score(y, pred):.4f} | "
          f"Recall {recall_score(y, pred):.4f} | Spec {tn / (tn + fp):.4f} | "
          f"F1 {f1_score(y, pred):.4f} | AUC {roc_auc_score(y, p):.4f}")
    print(f"Confusion [[TN FP][FN TP]]: [[{tn}, {fp}], [{fn}, {tp}]]")

report(test_df["label"].values, test_prob, best_t, "Test, frame level")

test_df["prob"] = test_prob
vid = test_df.groupby("video").agg(prob=("prob", "mean"), label=("label", "first"),
                                   category=("category", "first")).reset_index()
val_vid = val_df.assign(prob=val_prob).groupby("video").agg(prob=("prob", "mean"), label=("label", "first"))
vt = ts[np.argmax([f1_score(val_vid["label"], val_vid["prob"] > t) for t in ts])]
report(vid["label"].values, vid["prob"].values, vt, f"Test, video level (threshold {vt:.2f})")

print("\nVideo-level recall per fake method:")
vid["pred"] = (vid["prob"] > vt).astype(int)
print(vid[vid.label == 1].groupby("category")["pred"].mean().round(3))

In [ ]:
fpr, tpr, _ = roc_curve(test_df["label"], test_prob)
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(test_df['label'], test_prob):.3f}")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.title("ROC (test, frame level)")
plt.show()

model.save("/content/deepfake_effnetb0.keras")